# TinyDoc-VLM — Full 768 Retrain on Colab T4

Full-model (no LoRA) retrain at **768x768** with markdown-conversion synthetic data
(50K docs) + real benchmarks (OCRBench / FUNSD / CORD).

This is Tier-2 of the improvement plan (docs/model_improvements.md): after removing
the broken multi-task heads (A), raising resolution 384->768 (B), and adding the ngram
repetition penalty (C), this retrain is the "good margin" jump.

**Runtime:** T4 GPU (15GB). Change runtime type -> T4 -> Restart. ~1-2h data gen + ~8-16h train.

**Resumable:** every stage is guarded — if its output already exists it is skipped, so
you can hit Runtime -> Run all again after a disconnect and it continues where it left off.

## 1. Mount Google Drive (for checkpoint persistence)

In [ ]:
from google.colab import drive
import os

# Everything (repo, dataset, checkpoints) lives under WORK so it SURVIVES a
# Colab restart. If Drive mounts, WORK is on Drive; otherwise it is local
# (/content) and will be wiped on restart — re-run from scratch in that case.
try:
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/tinydoc-vlm'
    print('Google Drive mounted — work will persist across restarts.')
except Exception as e:
    print(f'Drive mount failed ({e}); using local dir (wiped on restart).')
    WORK = '/content/tinydoc-vlm'

os.makedirs(WORK, exist_ok=True)
print(f'WORK = {WORK}')

## 2. Clone repo & install deps

In [ ]:
import os, sys, subprocess, requests, zipfile, io, shutil

REPO_DIR = f'{WORK}/tinydoc-vlm'

if not os.path.exists(f'{REPO_DIR}/data/synthetic/markdown_dataset.py'):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print('Downloading repo via zip...')
    r = requests.get('https://github.com/eulogik/TinyDoc-VLM/archive/refs/heads/main.zip', timeout=60)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(WORK)
    os.rename(f'{WORK}/TinyDoc-VLM-main', REPO_DIR)

assert os.path.exists(f'{REPO_DIR}/data/synthetic/markdown_dataset.py'), 'Download failed'
os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], cwd=REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'sentencepiece', 'tokenizers', 'pillow', 'numpy',
                'pandas', 'tqdm', 'pyyaml', 'einops', 'faker', 'jinja2', 'pydantic',
                'datasets', 'accelerate'], cwd=REPO_DIR)
print('Repo ready — deps installed')

## 3. Generate 50K markdown-conversion dataset + real benchmarks

- data/synthetic/markdown_dataset.py renders synthetic docs -> prompt-target pairs
  (Convert the document to markdown:, Extract all text:, VQA, JSON).
- Real benchmarks are pulled from HuggingFace datasets into evaluation/data.
- data/build_training_dataset.py merges them into data/training/manifest.jsonl.

Lower NUM_DOCS (e.g. 15000) for a faster pilot. Generation runs on CPU (~1-2h for 50K).
If data/training/manifest.jsonl already exists with enough pairs, this stage is skipped.

In [ ]:
import os, sys, subprocess, shlex

REPO = f'{WORK}/tinydoc-vlm'
NUM_DOCS = 50000   # reduce to 15000 for a faster pilot
MANIFEST = 'data/training/manifest.jsonl'

def manifest_ok():
    if not os.path.exists(MANIFEST):
        return False
    n = sum(1 for _ in open(MANIFEST))
    return n >= 10000   # synthetic alone yields >10000 pairs for 50K docs

def run_streaming(cmd):
    print('$', shlex.join(cmd))
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')   # live progress, no buffering until the end
    p.wait()
    assert p.returncode == 0, f'Command failed (exit {p.returncode}): {shlex.join(cmd)}'

if manifest_ok():
    print(f'Manifest already exists ({sum(1 for _ in open(MANIFEST))} pairs) — skipping generation.')
else:
    # Only the 3 benchmarks actually used for training (OCRBench / FUNSD / CORD).
    # DocVQA (huge) and SROIE (no GT) are skipped by default.
    run_streaming([sys.executable, 'evaluation/download_benchmarks.py',
                   '--data-dir', 'evaluation/data',
                   '--benchmarks', 'ocrbench', 'funsd', 'cord'])
    print(f'Generating {NUM_DOCS} synthetic markdown docs + merging real benchmarks...')
    run_streaming([sys.executable, 'data/build_training_dataset.py',
                   '--num-docs', str(NUM_DOCS),
                   '--output-dir', 'data/training',
                   '--data-dir', 'evaluation/data'])
    assert manifest_ok(), 'Training data generation failed'

print(f'Combined training pairs: {sum(1 for _ in open(MANIFEST))}')

## 4. Initialize 768 model

training/init_768_model.py builds a 768 model from the 384 base checkpoint: copies the
trained decoder + compressor and interpolates the vision positional embeddings 384->768.
Skipped if checkpoints/init_768/config.json already exists.

In [ ]:
import os, sys, subprocess

REPO = f'{WORK}/tinydoc-vlm'
os.chdir(REPO)
if os.path.exists('checkpoints/init_768/config.json'):
    print('init_768 already exists — skipping.')
else:
    r = subprocess.run([sys.executable, 'training/init_768_model.py',
                        '--base', 'eulogik/TinyDoc-VLM-256M',
                        '--out', 'checkpoints/init_768'], cwd=REPO,
                       capture_output=True, text=True)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2500:])
print('init_768 ready:', os.path.exists('checkpoints/init_768/config.json'))

## 5. Full fine-tune at 768 on T4

Trains ALL parameters (no LoRA) with bf16 + gradient checkpointing.
~0.5-1 step/s on T4 -> 30000 steps ~ 8-16h. Checkpoints saved locally; only the final
one is synced to Drive (intermediate checkpoints are throttled via --save-every to avoid
filling the disk). Skipped if the final checkpoint already exists.

In [ ]:
import os, sys, subprocess, shlex, shutil
from pathlib import Path

REPO = f'{WORK}/tinydoc-vlm'
os.chdir(REPO)
STEPS = 30000
OUT = 'checkpoints/full768'
FINAL = Path(OUT) / 'final'

def run_streaming(cmd):
    print('$', shlex.join(cmd))
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')   # live progress
    p.wait()
    return p.returncode

if FINAL.exists():
    print('Final checkpoint already exists — skipping training.')
else:
    rc = run_streaming([sys.executable, 'training/full_train.py',
                        '--model-id', 'checkpoints/init_768',
                        '--manifest', 'data/training/manifest.jsonl',
                        '--steps', str(STEPS),
                        '--batch-size', '4',
                        '--grad-accum', '8',
                        '--warmup', '500',
                        '--lr', '1e-4',
                        '--save-every', '10000',
                        '--device', 'cuda',
                        '--bf16',
                        '--grad-checkpoint',
                        '--output-dir', OUT,
                        '--max-samples', '2000000'])
    if rc != 0:
        print(f'Training failed (exit {rc}) — see log above.')

if FINAL.exists():
    dst = Path(f'{WORK}/checkpoints/full768_final')
    shutil.copytree(FINAL, dst, dirs_exist_ok=True)
    print(f'Synced final checkpoint to {dst}')
else:
    print('Final checkpoint not found — check logs above.')

## 6. (Optional) Push trained model to Hugging Face

Paste your HF write token (Settings -> Access tokens). The pushed model is a full 768
TinyDoc-VLM (loadable with from_pretrained). It is published to a NEW repo
`eulogik/TinyDoc-VLM-768` to avoid overwriting the legacy `eulogik/TinyDoc-VLM-256M`.

In [ ]:
from huggingface_hub import login, HfApi
# Paste YOUR Hugging Face write token (Settings -> Access tokens) below.
HF_TOKEN = "hf_xxx"   # <-- replace with your token
login(token=HF_TOKEN)
HfApi().upload_folder(
    folder_path="checkpoints/full768/final",
    repo_id="eulogik/TinyDoc-VLM-768",   # NEW repo, do not overwrite the 256M base
    repo_type="model",
)
print("Pushed to Hub")